# Multi-height wind data

In this example we explore the calculation of multi-height wind data, as for example obtained from WRF results or downloaded from the [NEWA website](https://map.neweuropeanwindatlas.eu/) at a single point.

Here we will use the static data file `WRF-Timeseries-3000.nc` that is part of the `foxes` static data.
The time coordinate marks one month in 10 minute steps, and the wind speed (WS), wind direction (WD) and turbulent intensity (TI) are provided at 8 heights between 50 and 500 m. The air density (RHO) does not have height dependency but varies with time. The detailed data structure will be displayed below.

The basic assumption of this example is that we can calculate our wind farm results based on this data, i.e., that the horizontal variation can be neglected (for completely heterogeneous inflow data, see the corresponding example).

These are the imports for this example:

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from xarray import open_dataset

import foxes
import foxes.variables as FV

We use the default engine for the computations in this example:

In [ ]:
engine = foxes.Engine.new("default", n_procs=4)

First, we setup the model book and the wind farm. We choose 5 turbines in a row:

In [ ]:
# create wind farm, a single row of turbines:
farm = foxes.WindFarm()
foxes.input.farm_layout.add_row(
    farm=farm,
    xy_base=[0.0, 0.0],
    xy_step=[600.0, 0.0],
    n_turbines=5,
    turbine_models=["NREL5MW"],
    H=200.0,
    verbosity=0,
)

ax = foxes.algorithms.Downwind(
    farm,
    foxes.input.states.SingleStateStates(ws=8, wd=270, ti=0.08, rho=1.225),
    wake_models=[],
    verbosity=0,
).calc_outputs(
    outputs=foxes.models.outputs.FarmLayoutOutput(
        operation="get_figure", parameters=dict(figsize=(5, 3))
    )
)
plt.show()

Note that we manually change the hub height from 90 m to 200 m here. Next, we create the states based on the static data file `WRF-Timeseries-3000.nc`:

In [ ]:
# This is what the nc file looks like:
fpath = foxes.StaticData().get_file_path(foxes.STATES, "WRF-Timeseries-3000.nc")
with open_dataset(fpath) as ds:
    print(ds)

Now let's create the corresponding states object:

In [ ]:
states = foxes.input.states.MultiHeightNCTimeseries(
    data_source="WRF-Timeseries-3000.nc",
    time_coord="Time",
    h_coord="height",
    output_vars=[FV.WS, FV.WD, FV.TI, FV.RHO],
    var2col={FV.WS: "ws", FV.WD: "wd", FV.TI: "ti", FV.RHO: "rho"},
)

with engine:
    rose_farm = foxes.WindFarm()
    rose_farm.add_turbine(
        foxes.Turbine([0.0, 0.0], H=100.0, D=1.0, turbine_models=["null_type"]),
        verbosity=0,
    )
    foxes.algorithms.Downwind(
        rose_farm, states, wake_models=[], verbosity=0
    ).calc_outputs(
        outputs=foxes.models.outputs.StatesRosePlotOutput(
            point=[0.0, 0.0, 100.0], plot=True, plot_pars=dict(figsize=(6, 6))
        )
    )[0]
plt.show()

Note how the `var2col` option offers a mapping from the expected to the actual column names, if needed.

Let's next setup our algorithm. Notice that we include the z-sensitive rotor model `level10`, with 10 points on a vertical line (also the `grid` models would be an option). The partial wakes choice `None` represents default settings for all wake models. It is important that we do not select `rotor_points` together with the `level10` rotor, since averaging over a vertical line of points does not make much sense.

In [ ]:
algo = foxes.algorithms.Downwind(
    farm,
    states,
    rotor_model="level10",
    wake_models=["Bastankhah2014_linear_k1_02"],
    partial_wakes=None,
    verbosity=0,
)

We now visualize the vertical flow profile for a single selected state:

In [ ]:
with engine:
    farm_results = algo.calc_farm(outputs="farm_results_all")

In [ ]:
fr = farm_results.to_dataframe()
print(fr[[FV.WD, FV.REWS, FV.P]])

In [ ]:
with engine:
    plot_data = algo.calc_outputs(
        outputs=foxes.models.outputs.FlowPlots2D(
            plane="xz",
            reduction="states",
            plot=False,
            data_format="numpy",
            var=FV.AMB_WS,
            states_sel=["2009-01-06 13:50:00"],
            grid_pars=dict(
                resolution=10,
                x_direction=270,
                xmin=0.0,
                xmax=1000.0,
                zmin=50.0,
                zmax=500.0,
            ),
        ),
        farm_results=farm_results,
    )

g = algo.calc_outputs(
    outputs=foxes.models.outputs.FlowPlots2D(
        plane="xz",
        reduction="states",
        plot=True,
        data_format="numpy",
        var=FV.AMB_WS,
        states_sel=["2009-01-06 13:50:00"],
        grid_pars=dict(
            resolution=10, x_direction=270, xmin=0.0, xmax=1000.0, zmin=50.0, zmax=500.0
        ),
        plot_pars=dict(figsize=(8, 6)),
    ),
    farm_results=farm_results,
)
fig = g[0]
plt.show()

Let's look at the mean REWS results for each turbine:

In [ ]:
foxes.algorithms.Downwind(
    farm,
    foxes.input.states.SingleStateStates(ws=8, wd=270, ti=0.08, rho=1.225),
    wake_models=[],
    verbosity=0,
).calc_outputs(
    outputs=foxes.models.outputs.FarmLayoutOutput(
        operation="get_figure",
        parameters=dict(
            color_by="mean_REWS", title="Mean REWS [m/s]", s=150, annotate=0
        ),
    ),
    farm_results=farm_results,
)
plt.show()